# 1. Vanilla RNN — 한 timestep의 순전파

Vanilla RNN의 한 timestep에서는 현재 입력 `x_t`와 이전 hidden state `h_{t-1}`를 함께 사용해 현재 hidden state `h_t`를 만듭니다.

$$
net_t = W_{xh}x_t + W_{hh}h_{t-1} + b_h
$$

$$
h_t = \tanh(net_t)
$$

- `W_xh`: 입력 `x_t`를 hidden 공간으로 보내는 가중치
- `W_hh`: 이전 hidden state `h_{t-1}`를 현재 hidden state 계산에 사용하는 recurrent weight
- `net_t`: 활성화 함수에 들어가기 전 값
- `h_t`: 현재 timestep의 hidden state

1997년 LSTM 논문의 고전 표기 `net_i(t)`, `y_i(t)`를 현대적인 벡터 표기로 묶어서 쓴 형태라고 보면 됩니다.

In [1]:
import numpy as np

input_size = 2
hidden_size = 3

W_xh = np.array([
    [0.2, -0.1],
    [0.4,  0.3],
    [-0.5, 0.2],
])

W_hh = np.array([
    [0.5, 0.1, 0.0],
    [0.0, 0.4, 0.2],
    [0.1, 0.0, 0.3],
])

b_h = np.zeros(hidden_size)

def rnn_step(x_t, h_prev):
    net_t = W_xh @ x_t + W_hh @ h_prev + b_h
    h_t = np.tanh(net_t)
    return net_t, h_t

x_t = np.array([1.0, 0.5])
h_prev = np.zeros(hidden_size)

net_t, h_t = rnn_step(x_t, h_prev)

print('x_t shape     :', x_t.shape)
print('h_prev shape  :', h_prev.shape)
print('W_xh shape   :', W_xh.shape)
print('W_hh shape   :', W_hh.shape)
print('net_t         :', net_t)
print('h_t           :', h_t)


x_t shape     : (2,)
h_prev shape  : (3,)
W_xh shape   : (3, 2)
W_hh shape   : (3, 3)
net_t         : [ 0.15  0.55 -0.4 ]
h_t           : [ 0.14888503  0.50052021 -0.37994896]


### 코드 설명

`x_t.shape == (2,)`이고 `W_xh.shape == (3, 2)`이므로

$$
W_{xh}x_t \in \mathbb{R}^{3}
$$

가 됩니다. 마찬가지로 `W_hh.shape == (3, 3)`이고 `h_prev.shape == (3,)`이므로

$$
W_{hh}h_{t-1} \in \mathbb{R}^{3}
$$

입니다. 둘을 더하면 hidden unit 3개의 활성화 이전 값 `net_t`가 만들어지고, `tanh`를 적용하면 `h_t`가 됩니다.

즉 `net_t`와 `h_t`는 **벡터 전체**이고, 논문의 `net_i(t)`와 `y_i(t)`는 그 벡터에서 **i번째 unit 하나의 스칼라 값**을 가리킵니다.

# 2. Vanilla RNN — 여러 timestep으로 펼쳐 보기

이제 같은 `W_xh`, `W_hh`를 매 timestep 반복해서 사용해 봅니다.

RNN을 시간축으로 펼치면 다음과 같이 볼 수 있습니다.

```text
x_1       x_2       x_3       x_4
 ↓         ↓         ↓         ↓
h_1  ─→   h_2  ─→   h_3  ─→   h_4
```

여기서 중요한 점은 `W_hh^(1)`, `W_hh^(2)`처럼 timestep마다 새로운 recurrent weight가 생기는 것이 아니라, **동일한 `W_hh`가 모든 timestep에서 재사용된다는 것**입니다.

이 weight sharing 때문에 역전파 시에도 같은 recurrent transformation의 미분이 여러 번 연속해서 곱해지고, 이후 vanishing / exploding gradient 문제로 이어집니다.

In [2]:
sequence = np.array([
    [1.0,  0.5], # x_1
    [0.2, -0.3], # x_2
    [0.8,  0.1], # x_3
    [-0.4, 0.7], # x_4
])

h_t = np.zeros(hidden_size)

hidden_states = []
net_states = []

for t, x_t in enumerate(sequence, start=1):
    net_t, h_t = rnn_step(x_t, h_t)
    net_states.append(net_t.copy())
    hidden_states.append(h_t.copy())

    print(f't={t}')
    print('  x_t   =', x_t)
    print('  net_t =', net_t)
    print('  h_t   =', h_t)
    print()

hidden_states = np.array(hidden_states)

print('sequence shape      :', sequence.shape)
print('hidden_states shape :', hidden_states.shape)


t=1
  x_t   = [1.  0.5]
  net_t = [ 0.15  0.55 -0.4 ]
  h_t   = [ 0.14888503  0.50052021 -0.37994896]

t=2
  x_t   = [ 0.2 -0.3]
  net_t = [ 0.19449454  0.11421829 -0.25909619]
  h_t   = [ 0.19207863  0.11372418 -0.25344997]

t=3
  x_t   = [0.8 0.1]
  net_t = [ 0.25741173  0.34479968 -0.43682713]
  h_t   = [ 0.25187305  0.33175559 -0.41101101]

t=4
  x_t   = [-0.4  0.7]
  net_t = [0.00911208 0.10050004 0.241884  ]
  h_t   = [0.00911183 0.10016304 0.23727448]

sequence shape      : (4, 2)
hidden_states shape : (4, 3)


### 코드 설명

`sequence.shape == (4, 2)`는 **4개의 timestep × 각 timestep마다 입력 feature 2개**라는 뜻입니다.

각 timestep에서 만들어지는 `h_t`는 길이 3의 벡터이므로, 모든 hidden state를 쌓으면

$$
[h_1, h_2, h_3, h_4]
$$

가 모여 `hidden_states.shape == (4, 3)`이 됩니다.

그리고 다음 timestep에서는 직전에 계산된 `h_t`가 그대로 `h_{t-1}` 역할을 합니다.

```python
net_t, h_t = rnn_step(x_t, h_t)
```

이 한 줄에서 오른쪽의 `h_t`는 **이전 timestep의 hidden state**, 왼쪽의 `h_t`는 **새로 계산된 현재 hidden state**입니다.

다음 단계에서는 이 연결 하나를 미분해서

$$
\frac{\partial h_t}{\partial h_{t-1}}
$$

가 왜 `W_hh`와 `tanh`의 미분을 포함하는지 확인하고, 그 행렬이 바로 왜 Jacobian인지 연결합니다.

# 3. Jacobian — $\frac{\partial h_t}{\partial h_{t-1}}$ 직접 계산하기

Vanilla RNN의 한 timestep을 다시 쓰면 다음과 같습니다.

$$
net_t = W_{xh}x_t + W_{hh}h_{t-1} + b_h
$$

$$
h_t = \tanh(net_t)
$$

## 첫번째 미분 $\frac{\partial h_t}{\partial net_t}$ 구하기

위 식은 아래와 같이 계산 됩니다. 

각각 값이 아래와 같을 때

$$
net_t=
\begin{bmatrix}
n_1\\
n_2\\
n_3
\end{bmatrix}
\ \ \ \ 
h_t=
\begin{bmatrix}
\tanh(n_1)\\
\tanh(n_2)\\
\tanh(n_3)
\end{bmatrix}
$$

아래와 같이 계산됩니다. 여기에서 $\frac{\partial h_1}{\partial n_2}=0$ 같이 관계 없는 요소끼리는 결과가 0입니다.

$$
\frac{\partial h_t}{\partial net_t}
=
\begin{bmatrix}
\frac{\partial h_1}{\partial n_1} &
\frac{\partial h_1}{\partial n_2} &
\frac{\partial h_1}{\partial n_3}\\
\frac{\partial h_2}{\partial n_1} &
\frac{\partial h_2}{\partial n_2} &
\frac{\partial h_2}{\partial n_3}\\
\frac{\partial h_3}{\partial n_1} &
\frac{\partial h_3}{\partial n_2} &
\frac{\partial h_3}{\partial n_3}
\end{bmatrix}
=
\begin{bmatrix}
\tanh'(n_1)&0&0\\
0&\tanh'(n_2)&0\\
0&0&\tanh'(n_3)
\end{bmatrix}
$$

우리가 알고 싶은 것은 **이전 hidden state의 각 원소가 현재 hidden state의 각 원소에 얼마나 영향을 주는가**입니다.

$h_{t-1}$과 $h_t$는 둘 다 벡터이므로, 미분 결과는 숫자 하나가 아니라 행렬이 됩니다. 이 행렬을 **Jacobian**이라고 합니다.

$$
J_t = \frac{\partial h_t}{\partial h_{t-1}}
$$

chain rule을 적용하면

$$
\frac{\partial h_t}{\partial h_{t-1}}
=
\frac{\partial h_t}{\partial net_t}
\frac{\partial net_t}{\partial h_{t-1}}
$$

## 두번째 미분 $\frac{\partial net_t}{\partial h_{t-1}}$ 구하기

$$
net_t=W_{hh}h_{t-1}+\text{(h와 무관한 항들)}
$$ 

일 때

$$

W_{hh}=
\begin{bmatrix}
w_{11}&w_{12}\\
w_{21}&w_{22}
\end{bmatrix}
\ \ \ \ 
h_{t-1}=
\begin{bmatrix}
h_1\\
h_2
\end{bmatrix}
$$

$$
net_t=
\begin{bmatrix}
w_{11}h_1+w_{12}h_2\\
w_{21}h_1+w_{22}h_2
\end{bmatrix}
$$

가 되며 이를 이용하여 

$$
net_t=
\begin{bmatrix}
w_{11}h_1+w_{12}h_2\\
w_{21}h_1+w_{22}h_2
\end{bmatrix}
$$

$$
\frac{\partial net_t}{\partial h_{t-1}}
=
\begin{bmatrix}
w_{11}&w_{12}\\
w_{21}&w_{22}
\end{bmatrix}
=
W_{hh}
$$

가 됩니다.

## 마무리 - 두 연산 곱

$$
\frac{\partial net_t}{\partial h_{t-1}} = W_{hh}
$$

$$
\frac{\partial h_t}{\partial net_t}=\operatorname{diag}(1-\tanh^2(net_t))
$$

이므로

$$
\boxed{J_t = \operatorname{diag}(1-h_t^2)W_{hh}}
$$

hidden state가 weight가 되는 것이 아니라, $h_{t-1} \rightarrow net_t$ 구간을 미분했을 때 recurrent weight $W_{hh}$가 등장합니다.

> $\operatorname{diag}$ 는 벡터를 대각 행렬의 형태로 만드는 연산을 의미합니다.

In [ ]:
# t=2에서 Jacobian을 직접 계산해 보기
x_1 = sequence[0]
x_2 = sequence[1]

h_0 = np.zeros(hidden_size)
# x1하고 hidden state -> net_1 -> tanh(net_1) 같은 형식
net_1, h_1 = rnn_step(x_1, h_0)
net_2, h_2 = rnn_step(x_2, h_1)

# tanh'(z) = 1 - tanh(z)^2 - net_t가 변함에 따른 h_t 의 변화율
# shape = 1 - [3] 
tanh_prime = 1 - h_2**2
D_t = np.diag(tanh_prime)

# J_t = ∂h_t / ∂h_{t-1}
J_t = D_t @ W_hh

print('h_1 =', h_1)
print('net_2 =', net_2)
print('h_2 =', h_2)
print('tanh_prime =', tanh_prime)
print('\nD_t =')
print(D_t)
print('\nJacobian J_t =')
print(J_t)
print('\nJ_t shape =', J_t.shape)


h_1 = [ 0.14888503  0.50052021 -0.37994896]
net_2 = [ 0.19449454  0.11421829 -0.25909619]
h_2 = [ 0.19207863  0.11372418 -0.25344997]
tanh_prime = [0.9631058  0.98706681 0.93576311]

D_t =
[[0.9631058  0.         0.        ]
 [0.         0.98706681 0.        ]
 [0.         0.         0.93576311]]

Jacobian J_t =
[[0.4815529  0.09631058 0.        ]
 [0.         0.39482672 0.19741336]
 [0.09357631 0.         0.28072893]]

J_t shape = (3, 3)


### 코드 설명

J_t의 shape은 (3, 3)입니다. hidden state가 3차원 벡터이기 때문입니다.

예를 들어

$$
J_t[0,1]=\frac{\partial h_{t,0}}{\partial h_{t-1,1}}
$$

은 이전 timestep의 두 번째 hidden unit을 아주 조금 바꾸었을 때 현재 timestep의 첫 번째 hidden unit이 얼마나 변하는지를 뜻합니다.

계산 경로는

```text
h_{t-1} -- W_hh --> net_t -- tanh --> h_t
```

이고, 역전파에서는 이 경로를 반대로 따라가면서 각 구간의 미분을 곱합니다.

따라서 한 timestep을 거슬러 갈 때 gradient에 Jacobian이 한 번 곱해지고, 여러 timestep을 거슬러 가면

$$
J_tJ_{t-1}J_{t-2}\cdots
$$

처럼 반복해서 곱해집니다.

### 지금까지의 해석 정리

사용자 관점으로 정리하면, Jacobian은 **과거 timestep에서 생긴 hidden-state 변화의 영향을 마지막 hidden state까지 전달해 주는 연산**이라고 볼 수 있습니다.

예를 들어 마지막 hidden state가 $h_n$이고, 과거 어떤 timestep $t$에서 같은 recurrent weight $W_{hh}$가 사용되었다면 그 timestep의 local weight 영향은

$$
\frac{\partial h_t}{\partial W_{hh}}
$$

입니다. 이 영향이 마지막 hidden state $h_n$까지 가려면 중간 timestep들의 Jacobian을 통과합니다.

$$
J_nJ_{n-1}\cdots J_{t+1}\frac{\partial h_t}{\partial W_{hh}}
$$

그리고 마지막으로 loss가 $h_n$에 얼마나 민감한지

$$
\frac{\partial E}{\partial h_n}
$$

를 곱하면, timestep $t$에서 사용된 $W_{hh}$가 최종 loss에 기여한 gradient가 됩니다.

$$
\frac{\partial E}{\partial h_n}
J_nJ_{n-1}\cdots J_{t+1}
\frac{\partial h_t}{\partial W_{hh}}
$$

같은 $W_{hh}$가 모든 timestep에서 재사용되므로, 최종 $W_{hh}.grad$에는 각 timestep에서 생긴 이런 항들이 모두 더해집니다.

$$
\frac{\partial E}{\partial W_{hh}}
=
\sum_t
\frac{\partial E}{\partial h_n}
J_nJ_{n-1}\cdots J_{t+1}
\frac{\partial h_t}{\partial W_{hh}}
$$

따라서 핵심은 다음과 같습니다.

- $\frac{\partial h_t}{\partial W_{hh}}$: 그 timestep에서 $W_{hh}$가 $h_t$에 준 직접적인 영향
- $J_nJ_{n-1}\cdots J_{t+1}$: 그 영향을 마지막 hidden state까지 전달하는 과정
- $\frac{\partial E}{\partial h_n}$: 마지막 hidden state 변화가 loss에 미치는 영향
- 위 항들을 timestep별로 모두 더한 것이 $W_{hh}.grad$

즉 vanishing / exploding은 **gradient끼리 직접 곱해서 생기는 것이 아니라**, gradient를 뒤로 전달하는 Jacobian들이 반복해서 곱해지면서 과거 timestep의 영향이 지나치게 작아지거나 커지는 현상입니다.

다음 단계에서는 이 반복 곱이 실제로 gradient vanishing / exploding을 어떻게 만드는지 숫자로 확인합니다.



# 4. Gradient Vanishing / Exploding — Jacobian 누적곱으로 확인하기

이제 여러 timestep의 Jacobian을 실제로 곱해 보면서, 과거의 영향이 마지막 hidden state까지 전달될 때 얼마나 줄어들거나 커지는지 확인합니다.

각 timestep에서

$$
J_t = \frac{\partial h_t}{\partial h_{t-1}}
$$

이고, $h_1$의 변화가 $h_4$까지 전달되는 정도는

$$
\frac{\partial h_4}{\partial h_1} = J_4J_3J_2
$$

로 볼 수 있습니다.

중요한 점은 **gradient 자체를 서로 곱하는 것이 아니라**, gradient를 시간축 뒤로 전달하는 Jacobian들이 반복해서 곱해진다는 것입니다.


In [ ]:
def forward_with_jacobians(sequence, W_hh_used):
    """
    자코비안 식을 이용해서 모든 sequence에 대해 timestep를 하는 과정입니다.
    입력 값으로는 사용되는 W_hh 행렬을 받으며 해당 값으로 새로 만들 sequence를 받아줍니다.
    """

    # h_0을 위한 값을 만들어줍니다.
    h_prev = np.zeros(hidden_size)
    # 매 timestep마다 h_t와 J_t 를 만들어줍니다
    jacobians = []
    hidden_list = []

    # 입력은 sequence에서 하나씩 뽑아와줍니다.
    for x_t in sequence:
        # 다음 h_t를 구해줍니다. x_1 이라면 h_1을 구해줍니다.
        net_t = W_xh @ x_t + W_hh_used @ h_prev + b_h
        h_t = np.tanh(net_t)

        # J_t = diag(tanh'(net_t)) @ W_hh
        # 아래 수식의 `1 - h_t**2`는 [d tanh(net_t) / d net_t] 미분값입니다.
        D_t = np.diag(1 - h_t**2)
        # 해당 값을 W_hh 에 연산해주면 자코비안 값이 완성됩니다.
        # D_t는 `d[h_t]/d[net_t]`를 의미합니다.
        # Whh는 `d[net_t]/d[h_{t-1}]`을 의미합니다.
        J_t = D_t @ W_hh_used

        # 자코비안 결과를 마지막 항부터 누적시켜줍니다.
        jacobians.append(J_t)
        # h_t 저장
        hidden_list.append(h_t.copy())
        
        # 다음 입력을 받아주기 위해 h_{t-1} 저장해주기
        h_prev = h_t

    return hidden_list, jacobians


hidden_list, jacobians = forward_with_jacobians(sequence, W_hh)

J_1, J_2, J_3, J_4 = jacobians

# h_1의 변화가 h_4까지 전달되는 정도
J_4_to_1 = J_4 @ J_3 @ J_2

print('||J_2|| =', np.linalg.norm(J_2, 2))
print('||J_3 J_2|| =', np.linalg.norm(J_3 @ J_2, 2))
print('||J_4 J_3 J_2|| =', np.linalg.norm(J_4_to_1, 2))

print('\nJ_4 J_3 J_2 =')
print(J_4_to_1)


||J_2|| = 0.5343609807425027
||J_3 J_2|| = 0.27038139855197896
||J_4 J_3 J_2|| = 0.1481421970991417

J_4 J_3 J_2 =
[[0.11440684 0.05508939 0.02126676]
 [0.01913841 0.05724015 0.06147158]
 [0.03921591 0.01001177 0.02156015]]


### 코드 설명

`np.linalg.norm(..., 2)`는 행렬이 어떤 방향의 벡터를 **최대로 얼마나 늘리거나 줄일 수 있는지**를 보는 값입니다.

따라서


$$\,\,\,\,\,\,\,\,J_2         = 0.5343609807425027 $$
$$\,\,\,\,\,J_3 J_2     = 0.27038139855197896 $$
$$J_4 J_3 J_2 = 0.1481421970991417 $$

를 순서대로 비교하면, timestep을 더 많이 거슬러 갈수록 변화량 전달 능력이 어떻게 변하는지 볼 수 있습니다.

현재 `W_hh`에서는 `tanh`의 미분값도 대부분 1 이하이고 recurrent weight도 크지 않으므로, 누적곱의 크기가 점점 작아지는 경향을 볼 수 있습니다. 이것이 **vanishing gradient** 쪽의 직관입니다.


## 4-1. 같은 구조에서 recurrent weight의 크기만 바꿔 보기

이번에는 구조와 입력은 그대로 두고 `W_hh`의 크기만 바꿉니다.

- 작은 recurrent weight: 과거 영향이 빠르게 줄어드는 경향
- 큰 recurrent weight: 특정 방향의 영향이 커질 가능성

단, 실제 RNN에서는 `tanh`의 미분도 함께 곱해지므로 단순히 `W_hh > 1`이라는 이유만으로 항상 exploding이 발생하는 것은 아닙니다.


In [5]:
def jacobian_product_norm(sequence, W_hh_used):
    """변형된 크기의 W_hh를 받아서 처리할 함수"""
    _, jacobians = forward_with_jacobians(sequence, W_hh_used)

    # h_1 -> h_4 전달: J_4 @ J_3 @ J_2
    product = np.eye(hidden_size) # 단위행렬 생성
    # 누적 Jacobian 크기 저장
    norms = []

    for J_t in jacobians[1:]:
        # 단위행렬에 곱하면 (A, B) @ (B, B) 에서 이면 동일한 (A, B)를 반환
        product = J_t @ product
        norms.append(np.linalg.norm(product, 2))

    return norms

# 작은 W_hh인 경우
W_small = W_hh * 0.5
W_original = W_hh.copy()
# 큰 W_hh인 경우
W_large = W_hh * 3.0

small_norms = jacobian_product_norm(sequence, W_small)
original_norms = jacobian_product_norm(sequence, W_original)
large_norms = jacobian_product_norm(sequence, W_large)

print('누적 단계     : 1-step, 2-step, 3-step')
print('small W_hh   :', small_norms)
print('original W_hh:', original_norms)
print('large W_hh   :', large_norms)


누적 단계     : 1-step, 2-step, 3-step
small W_hh   : [np.float64(0.27196858626463927), np.float64(0.07059509684167227), np.float64(0.019194849234581677)]
original W_hh: [np.float64(0.5343609807425027), np.float64(0.27038139855197896), np.float64(0.1481421970991417)]
large W_hh   : [np.float64(1.3943035463637188), np.float64(1.4744813468426914), np.float64(1.831998593154269)]


### 코드 설명

이 실험의 핵심은 정확한 숫자 자체보다 **누적 Jacobian의 크기가 timestep 수에 따라 어떤 방향으로 변하는지**입니다.

과거 timestep $t$의 local gradient가 마지막 timestep $n$의 loss까지 전달될 때는 대략

$$
\frac{\partial E}{\partial h_n}
J_nJ_{n-1}\cdots J_{t+1}
\frac{\partial h_t}{\partial W_{hh}}
$$

형태가 됩니다.

그래서 $J_nJ_{n-1}\cdots$의 크기가 반복해서 작아지면 과거 timestep의 기여가 거의 0이 되어 **vanishing gradient**, 반대로 반복해서 커지면 **exploding gradient**가 됩니다.

즉 이 문제의 본질은

```text
과거의 local gradient
    ↓
Jacobian
    ↓
Jacobian
    ↓
Jacobian
    ↓
마지막 loss
```

처럼 **시간축을 따라 반복되는 미분 변환**입니다.

다음 단계에서는 이 문제를 보고 1997년 LSTM이 왜 `weight = 1`인 특별한 self-connection을 만들려고 했는지, 즉 naive constant-error solution과 CEC로 넘어갑니다.
